# 개별종목 조합E — LightGBM

`기본모델/04.LightGBM.ipynb`과 같은 `models.lightgbm.build_lightgbm_baseline`을 가져오고
조합E 피처를 주입합니다. 기본모델 코드는 `models/`에 한 번만 존재합니다.
후보·라벨·날짜 그룹 12폴드 실행은 모든 조합이 같은 공통 함수를 사용합니다.


In [1]:
# 1. 기본모델을 가져옵니다.
import sys
from pathlib import Path

import pandas as pd
from IPython.display import display

project_root = Path.cwd().resolve()
while project_root != project_root.parent and not (project_root / "pyproject.toml").is_file():
    project_root = project_root.parent
if not (project_root / "pyproject.toml").is_file():
    raise RuntimeError("프로젝트 루트를 찾지 못했습니다.")
if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))

from models.lightgbm import build_lightgbm_baseline  # noqa: E402

MODEL_NAME = 'LightGBM'
MODEL_BUILDER = build_lightgbm_baseline


In [2]:
# 2. 조합E의 피처 값만 지정합니다.
import json

COMBINATION = 'E'
FEATURE_COLUMNS = (
    'sector_ret_5',
    'sector_ret_20',
    'relative_ret_5_sector',
    'relative_ret_20_sector',
    'relative_ret_5_market',
    'sector_hv_20',
    'sector_beta_60',
)

report_path = project_root / "reports" / "stock_feature_combinations.json"
report = json.loads(report_path.read_text(encoding="utf-8"))
combination_report = report["combinations"][COMBINATION]
panel = combination_report["panel"]
print("학습 기간:", panel["first_date"], "~", panel["last_date"])
print("학습 행·종목:", panel["model_rows"], panel["stocks"])
print(f"조합{COMBINATION} 피처:", FEATURE_COLUMNS)

folds = pd.DataFrame(combination_report["outer_fold_results"])
model_folds = folds.loc[folds["model"].eq(MODEL_NAME)].reset_index(drop=True)
fold_columns = [
    "fold",
    "selected_class_weight",
    "train_dates",
    "valid_start",
    "valid_end",
    "accuracy",
    "training_majority_baseline_accuracy",
    "accuracy_minus_training_majority_baseline",
    "macro_f1",
    "down_recall",
    "core_harmonic_mean",
]
display(model_folds.loc[:, fold_columns].round(4))

metric_columns = [
    "accuracy",
    "training_majority_baseline_accuracy",
    "accuracy_minus_training_majority_baseline",
    "macro_f1",
    "down_recall",
    "core_harmonic_mean",
]
display(model_folds.loc[:, metric_columns].mean().to_frame("OOS 폴드 평균").round(4))

# 24개 노트북이 각각 중복 학습하지 않도록 실제 fit은 공통 실행기에서 한 번 수행합니다.
print("재실행 명령: python scripts/run_stock_model_experiment.py")


학습 기간: 20100331 ~ 20240822
학습 행·종목: 172058 162
조합E 피처: ('sector_ret_5', 'sector_ret_20', 'relative_ret_5_sector', 'relative_ret_20_sector', 'relative_ret_5_market', 'sector_hv_20', 'sector_beta_60')


,fold,selected_class_weight,train_dates,valid_start,valid_end,accuracy,training_majority_baseline_accuracy,accuracy_minus_training_majority_baseline,macro_f1,down_recall,core_harmonic_mean
0,1,balanced,750,20130410,20130705,0.3847,0.3701,0.0146,0.3741,0.3212,0.3578
1,2,balanced,999,20140414,20140711,0.4385,0.4743,-0.0358,0.3340,0.1732,0.2715
2,3,balanced,1248,20150421,20150716,0.3606,0.3301,0.0305,0.3585,0.3336,0.3505
3,4,balanced,1496,20160422,20160719,0.3676,0.4107,-0.0431,0.3402,0.2297,0.2996
4,5,balanced,1745,20170424,20170721,0.3857,0.4177,-0.0320,0.3361,0.2583,0.3178
5,6,balanced,1994,20180503,20180731,0.3816,0.3908,-0.0092,0.3768,0.3702,0.3761
6,7,balanced,2243,20190514,20190806,0.3867,0.4612,-0.0745,0.3369,0.2072,0.2890
7,8,balanced,2492,20200518,20200807,0.3682,0.3144,0.0538,0.3576,0.4452,0.3867
8,9,balanced,2741,20210518,20210810,0.3914,0.4418,-0.0504,0.3566,0.2848,0.3382
9,10,balanced,2989,20220519,20220812,0.3450,0.3343,0.0107,0.3363,0.2794,0.3174


,OOS 폴드 평균
accuracy,0.3824
training_majority_baseline_accuracy,0.3846
accuracy_minus_training_majority_baseline,-0.0022
macro_f1,0.3560
down_recall,0.2993
core_harmonic_mean,0.3372


재실행 명령: python scripts/run_stock_model_experiment.py
